In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
load_dotenv(override=True)
# Constants 

MODEL_NAME = "gpt-5.4-mini"
HOW_MANY_SEARCHES = 5# Constants 


In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = (
    "Help a company with a warehouse at 440 9th Ave, New York, NY anticipate weather, events, "
    "and other disruptions over the next week that may impact inbound and outbound shipping operations."
)

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. The query concerns a warehouse at 440 9th Ave in Manhattan,
so prioritize searches for weather, local events, transit disruptions, road closures, and other
factors that could affect truck access and shipping near that location.
Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:
INSTRUCTIONS = f"""
You are a senior logistics analyst tasked with writing a cohesive report for this research query:
"{task}"

The company operates a warehouse at 440 9th Ave in Manhattan (Hell's Kitchen / West Midtown),
near the Lincoln Tunnel, West Side Highway, and Penn Station corridor. You will be provided
with the original query and summarized web research. Using that research, generate a comprehensive
report that helps the company anticipate disruptions to inbound and outbound shipping.

Organize findings by category (e.g. weather, transit disruptions, labor strikes, protests,
road/bridge closures, port or airport issues, and major public events). For each item identified,
include its expected timing, severity, and specific implications for truck access, loading docks,
and delivery routes serving 440 9th Ave (e.g. delivery delays, route closures, curb restrictions,
capacity constraints).

Conclude with a summary section highlighting the highest-priority risks the warehouse should
plan around this week, with practical recommendations for dispatch and scheduling.

Return structured output with:
- short_summary: a 2-3 sentence executive summary of the key findings
- markdown_report: the full detailed report in markdown (aim for 5-10 pages, at least 1000 words)
- follow_up_questions: suggested topics to research further if gaps remain in the research
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

In [ ]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

In [ ]:
import gradio as gr
import tempfile
from pathlib import Path
from datetime import datetime

def save_report_to_file(report) -> str:
    file_path = Path(tempfile.gettempdir()) / f"logistics_report_{datetime.now():%Y%m%d_%H%M%S}.md"
    content = (
        f"# Logistics Disruption Report\n\n"
        f"## Executive Summary\n\n{report.short_summary}\n\n---\n\n{report.markdown_report}"
    )
    file_path.write_text(content, encoding="utf-8")
    return str(file_path)

async def run_logi_agent():
    logs = []

    def log(msg: str) -> str:
        logs.append(msg)
        return "\n".join(logs)

    hidden_download = gr.update(value=None, visible=False)
    disabled_btn = gr.update(interactive=False)
    enabled_btn = gr.update(interactive=True)

    yield log("Starting research..."), "", hidden_download, disabled_btn

    query = task
    with trace("Research trace"):
        yield log("Planning searches..."), "", hidden_download, disabled_btn

        result = await Runner.run(planner_agent, f"Query: {query}")
        searches = result.final_output.searches
        yield log(f"Will perform {len(searches)} searches"), "", hidden_download, disabled_btn

        search_results = []
        for i, item in enumerate(searches, 1):
            yield log(f"Running search {i}/{len(searches)}: {item.query}"), "", hidden_download, disabled_btn
            search_results.append(await search(item))
            yield log(f"Completed search {i}/{len(searches)}"), "", hidden_download, disabled_btn

        yield log("Finished searching"), "", hidden_download, disabled_btn
        yield log("Thinking about report..."), "", hidden_download, disabled_btn

        report = await write_report(query, search_results)

        yield log("Finished writing report"), "", hidden_download, disabled_btn

    report_md = f"**Executive Summary:** {report.short_summary}\n\n---\n\n{report.markdown_report}"
    file_path = save_report_to_file(report)

    yield (
        log("Report complete!"),
        report_md,
        gr.update(value=file_path, visible=True),
        enabled_btn,
    )

with gr.Blocks(title="Research Agent Hub") as demo:
    gr.Markdown("# Research Agent Hub")
    gr.Markdown("Generate a shipping disruption report for the warehouse at 440 9th Ave.")
    run_btn = gr.Button("Run Logistics Agent", variant="primary")
    status_output = gr.Textbox(label="Progress", lines=10, interactive=False)
    report_output = gr.Markdown(label="Report")
    download_btn = gr.DownloadButton("Download Report", visible=False)

    run_btn.click(
        fn=run_logi_agent,
        outputs=[status_output, report_output, download_btn, run_btn],
        show_progress="full",
    )

demo.launch(inbrowser=True)